# Module 9 — Debugging challenge: worked troubleshooting

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/python_tutorial/09_debugging_challenge_solutions.ipynb)

> **Try the [challenge notebook](09_debugging_challenge.ipynb) first.** The point is practising how to *find* a bug. Reading the
> answer first skips exactly that.

For each challenge: the steps a careful debugger would take, the cause, the fixed code, and the same checks as
in the challenge (which now all print `PASS`). This notebook runs cleanly with *Run all*.

**The routine behind every fix:** reproduce → inspect intermediate values (shapes, types, a few numbers, a plot)
→ compare with something you know → name the cause → change one thing → verify with a check.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

fs = 250                                          # Hz, used by most challenges

def make_ecg(seconds=10, hr_bpm=72, fs=250, seed=0):
    """A synthetic ECG-like trace in mV: R and T waves, slow baseline wander and a little noise."""
    r = np.random.default_rng(seed)
    t = np.arange(int(seconds * fs)) / fs
    x = np.zeros(t.size)
    for beat in np.arange(0.3, seconds, 60 / hr_bpm):
        x += 1.00 * np.exp(-((t - beat) ** 2) / (2 * 0.012 ** 2))          # R wave
        x += 0.25 * np.exp(-((t - beat - 0.25) ** 2) / (2 * 0.04 ** 2))    # T wave
    x += 0.3 * np.sin(2 * np.pi * 0.25 * t)                               # baseline wander
    return x + 0.03 * r.standard_normal(t.size)

results = {}          # challenge number -> {requirement: True/False}, or None if its code didn't finish
current = None        # set by each check cell

def check(ok, what):
    """Print PASS / NOT YET for one requirement of the current challenge, and record it."""
    results.setdefault(current, {})[what] = bool(ok)
    print(("PASS     " if ok else "NOT YET  ") + what)

## Challenge 1 — The sample count ★

**Goal:** Compute how many samples a 10-second recording at `fs` has.

**Symptom:** The cell stops with an error.

**How to troubleshoot it**

1. **Reproduce.** Run the cell. It stops with `NameError: name 'sampling_rate' is not defined`.
2. **Read the error from the bottom.** The last line says *which* name is unknown; the arrow points at the line.
3. **Inspect.** Which names do exist? The setup cell defines `fs`, not `sampling_rate`. (`dir()` lists every defined name.)
4. **Cause.** A name used before it was defined — here a typo for an existing variable. The other common cause is a cell you haven't run yet.
5. **Fix** the name, re-run, and run the check.

**Fixed code**

In [ ]:
duration_s = 10
n_samples = int(duration_s * fs)
print(n_samples)

In [ ]:
current = 1
results[1] = {}
try:
    check(n_samples == 2500, "a 10-s recording at 250 Hz has 2500 samples")
except NameError:
    results[1] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Takeaway:** `NameError` means Python has never seen that name: a typo, or a cell you haven't run. Module 1 §7.

## Challenge 2 — The time axis ★

**Goal:** Plot the ECG against time in seconds. The recording lasts 10 s, so its last sample is at (N − 1)/fs = 9.996 s.

**Symptom:** The plot's x-axis runs to more than 600,000 seconds.

**How to troubleshoot it**

1. **Reproduce.** The plot looks like an ECG, but the x-axis is absurd.
2. **Inspect.** Print `t[:3]` and `t[-1]`: you get `0, 250, 500, ...` and `624750`. Each step should be 1/250 s = 0.004 s.
3. **Cause.** Sample *n* happens at time *n / fs*, not *n × fs*.
4. **Fix:** `t = np.arange(ecg.size) / fs`. The last sample is then at 9.996 s: the 10-s recording covers the interval up to, but not including, 10 s.

**Fixed code**

In [ ]:
ecg = make_ecg(fs=fs)
t = np.arange(ecg.size) / fs

fig, ax = plt.subplots(figsize=(9, 2.5))
ax.plot(t, ecg); ax.set_xlabel("time (s)"); ax.set_ylabel("amplitude (mV)")
plt.show()

In [ ]:
current = 2
results[2] = {}
try:
    check(t.size == ecg.size and t[0] == 0, "one time value per sample, starting at 0 s")
    check(np.allclose(np.diff(t), 1 / fs), "steps of 1/fs = 0.004 s")
    check(np.isclose(t[-1], (ecg.size - 1) / fs), "last sample at (N - 1)/fs = 9.996 s")
except NameError:
    results[2] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Takeaway:** Sanity-check units with a number you know: a 10-s recording must end just before 10 s. Module 2 §2.

## Challenge 3 — Per-channel RMS ★★

**Goal:** A two-channel recording arrives the way WFDB returns it, as `(n_samples, n_channels)`. Compute the RMS (root mean square) of each channel: two numbers.

**Symptom:** No error, but you get 2500 numbers instead of 2.

**How to troubleshoot it**

1. **Reproduce.** No error, which makes this bug dangerous: the wrong numbers flow on silently.
2. **Inspect shapes.** `rec.shape` is `(2500, 2)`; the function's docstring expects `(n_channels, n_samples)`. With `axis=1` it averaged across the 2 channels for every sample.
3. **Cause.** Two conventions for the same data: WFDB is samples × channels, MNE and the course are channels × samples.
4. **Fix:** pass `rec.T`. To make a wrong orientation fail loudly next time, check the shape against something you *know* — here the number of channels: `assert x.shape[0] == n_channels`. (A rule like "fewer rows than columns" is only a guess: 64 channels × 16 samples breaks it.)

**Fixed code**

In [ ]:
rec = np.c_[make_ecg(seed=1), 0.5 * make_ecg(seed=2)]       # (n_samples, n_channels), like WFDB

def rms_per_channel(x):
    """RMS of each channel. x must be (n_channels, n_samples)."""
    return np.sqrt(np.mean(x ** 2, axis=1))

rms = rms_per_channel(rec.T)                                  # transpose to (n_channels, n_samples)
print(rms.shape, rms)

In [ ]:
current = 3
results[3] = {}
try:
    reference = np.array([np.sqrt(np.mean(rec[:, k] ** 2)) for k in range(2)])   # RMS, one channel at a time
    check(np.shape(rms) == (2,), "one value per channel")
    check(np.shape(rms) == (2,) and np.allclose(rms, reference), "the values are the RMS of each channel")
except NameError:
    results[3] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Takeaway:** When numbers have the wrong *count*, print `.shape` at every step, and check shapes against known metadata. Modules 2 §4 and 3 §4.

## Challenge 4 — Scaling ADC counts ★★

**Goal:** The amplifier stored the ECG as 16-bit integers (`int16`). Each count is 30 µV. Convert the whole recording to **floating-point** values in µV, the representation all later processing expects. (The numbers are exaggerated so the problem is easy to see.)

**Symptom:** The R peaks look clipped, and some samples jump to large negative values.

**How to troubleshoot it**

1. **Reproduce.** Plot `ecg_uv`: the tops of the R peaks wrap around to large negative values.
2. **Inspect.** `ecg_uv.dtype` is `int16`, and `ecg_uv.max()` is about 32,700 — suspiciously close to 32,767, the largest `int16` (the range is −32,768 to 32,767). `raw.max() * 30` should be about 39,000.
3. **Cause.** Integer overflow: in NumPy 2, multiplying an `int16` array by a small Python integer keeps the result `int16`, and values outside the range wrap around silently.
4. **Fix:** convert to float first, `raw.astype(float) * gain`. (Widening to `int64` would also avoid the overflow, but later filtering needs floats anyway, which is why the goal asks for them.)

**Fixed code**

In [ ]:
raw = (make_ecg(seed=3) * 1000).astype(np.int16)             # ADC counts, as stored in the file
gain_uv_per_count = 30
ecg_uv = raw.astype(float) * gain_uv_per_count               # convert to float BEFORE scaling
print(ecg_uv.dtype, ecg_uv.min(), ecg_uv.max())

In [ ]:
current = 4
results[4] = {}
try:
    expected = raw.astype(np.float64) * 30                       # a safe reference, every sample
    check(np.issubdtype(ecg_uv.dtype, np.floating), "floating-point result")
    check(np.shape(ecg_uv) == raw.shape and np.allclose(ecg_uv, expected), "every sample equals counts × 30 µV")
except NameError:
    results[4] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Takeaway:** If a maximum sits right at a type's limit (32,767; 255; 65,535), suspect overflow. Modules 6 §5 and 8 §2.

## Challenge 5 — Removing baseline wander ★★

**Goal:** Remove the slow baseline wander (below 0.5 Hz) with a high-pass filter, keeping the R peaks (about 1 mV).

**Symptom:** The wander is gone — but so are the R peaks: the largest value is now about 0.07.

**How to troubleshoot it**

1. **Reproduce.** Plot before and after: the after-trace is tiny and spiky.
2. **Inspect the filter, not just the output.** `signal.sosfreqz(sos, fs=fs)` shows where it cuts off: around 62 Hz, not 0.5 Hz.
3. **Cause.** Without `fs=`, SciPy reads cut-offs as a fraction of the Nyquist frequency, so 0.5 meant 0.5 × 125 Hz = 62.5 Hz. The filter removed almost all of the ECG.
4. **Fix:** add `fs=fs`. Then plot the frequency response to confirm the cut-off is where you meant. Note the check has two parts: removing the wander alone is easy if you also destroy the signal.

**Fixed code**

In [ ]:
ecg5 = make_ecg(seed=4)
sos = signal.butter(2, 0.5, btype="highpass", fs=fs, output="sos")   # cut-off in Hz, because fs is given
ecg5_hp = signal.sosfiltfilt(sos, ecg5)
print(f"max before {ecg5.max():.2f}, after {ecg5_hp.max():.2f}")

In [ ]:
current = 5
results[5] = {}
try:
    f_chk, P_before = signal.welch(ecg5, fs=fs, nperseg=8 * fs)
    _, P_after = signal.welch(ecg5_hp, fs=fs, nperseg=8 * fs)
    low = f_chk <= 0.3
    check(P_after[low].sum() < 0.1 * P_before[low].sum(), "the slow wander is strongly reduced")
    check(ecg5_hp.max() > 0.8, "the R peaks are kept (maximum above 0.8 mV)")
except NameError:
    results[5] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Takeaway:** Pass `fs=` to every filter design and spectral function, and look at the frequency response. Modules 4 §3 and 8 §3.

## Challenge 6 — Heart rate from R peaks ★★

**Goal:** Find the R peaks and compute the heart rate in beats per minute. The true rate is 72 bpm.

**Symptom:** The heart rate comes out as 0.29 bpm.

**How to troubleshoot it**

1. **Reproduce.** 12 beats in 10 s is plausible for 72 bpm — a useful clue, but not proof that peak finding is right (a missed beat and a false one would cancel out). Check that the peak *positions* sit on the R waves, then look at what comes after.
2. **Inspect intermediate values.** `rr[:3]` shows about `208` — far too long for seconds, but exactly 0.83 s × 250 samples/s.
3. **Cause.** `find_peaks` returns sample *indices*. Their differences are in samples, not seconds.
4. **Fix:** divide by `fs`. Then 60 / 0.833 s = 72 bpm.

**Fixed code**

In [ ]:
ecg6 = make_ecg(seed=5)
ecg6 = signal.sosfiltfilt(signal.butter(2, 0.5, btype="highpass", fs=fs, output="sos"), ecg6)  # correct
peaks, _ = signal.find_peaks(ecg6, height=0.5, distance=int(0.3 * fs))
rr = np.diff(peaks) / fs                              # RR intervals in SECONDS
hr_bpm = 60 / np.mean(rr)
print(f"{peaks.size} beats, HR = {hr_bpm:.2f} bpm")

In [ ]:
current = 6
results[6] = {}
try:
    check(np.allclose(rr, np.diff(peaks) / fs), "RR intervals are in seconds")
    check(abs(hr_bpm - 72) < 1, "heart rate close to the true 72 bpm")
except NameError:
    results[6] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Takeaway:** Narrow a bug down by checking each intermediate value against something you know. Module 4 §4.

## Challenge 7 — A close-up of the first second ★★

**Goal:** Make `first_second`: the first second of the recording, centred on zero (mean removed), and plot it — **without changing the recording itself**.

**Symptom:** Later analysis of the full recording gives different numbers than before the close-up was made.

**How to troubleshoot it**

1. **Reproduce.** Compare `ecg7` with a freshly made copy: `np.array_equal(ecg7, make_ecg(seed=6))` is `False`.
2. **Find where it changed.** `np.flatnonzero(ecg7 != make_ecg(seed=6))` shows only the first 250 samples differ — exactly the slice.
3. **Cause.** A NumPy slice is a *view* of the same memory. `-=` changed the slice in place, and with it the original array.
4. **Fix:** take `.copy()` when you want to change a piece without touching the original. Writing `first_second = ecg7[:fs] - ecg7[:fs].mean()` works too: it creates a new array instead of changing one in place. Note the check has two parts: deleting the centring would also leave the recording unchanged, but miss the goal.

**Fixed code**

In [ ]:
ecg7 = make_ecg(seed=6)
first_second = ecg7[:fs].copy()                       # an independent copy, not a view
first_second -= first_second.mean()                   # centre it for the close-up plot
t7 = np.arange(fs) / fs
plt.figure(figsize=(6, 2)); plt.plot(t7, first_second)
plt.xlabel("time (s)"); plt.ylabel("amplitude (mV)"); plt.show()

In [ ]:
current = 7
results[7] = {}
try:
    original = make_ecg(seed=6)
    check(np.array_equal(ecg7, original), "the full recording is unchanged")
    check(np.size(first_second) == fs and np.allclose(first_second, original[:fs] - original[:fs].mean()),
          "first_second is the first second, centred on zero")
except NameError:
    results[7] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Takeaway:** If data changes when you didn't touch it, look for in-place operations (`-=`, `+=`, `x[...] = `) on slices. Module 8 §2.

## Challenge 8 — Downsampling for storage ★★★

**Goal:** A 10-s recording sampled at 500 Hz contains a 5 Hz rhythm (amplitude 1) and 60 Hz mains interference. Downsample it to 100 Hz (1000 samples) to save space, keeping the 5 Hz rhythm.

**Symptom:** The downsampled spectrum has a peak at 40 Hz that wasn't in the original.

**How to troubleshoot it**

1. **Reproduce.** The original has components at 5 and 60 Hz only, yet the new spectrum shows 5 and 40 Hz.
2. **Reason about it.** At 100 Hz the highest frequency you can represent (Nyquist) is 50 Hz. A 60 Hz component can't be represented, so it *folds back*: 100 − 60 = 40 Hz.
3. **Cause.** Aliasing: keeping every 5th sample without first removing everything above the new Nyquist frequency.
4. **Fix:** `signal.resample_poly(x, up=1, down=5)` or `signal.decimate(x, 5)`, which low-pass filter before downsampling; a low-pass filter of your own followed by `[::5]` is equally valid. Check all three parts: length, no alias, and the 5 Hz rhythm intact.

**Fixed code**

In [ ]:
fs8 = 500
t8 = np.arange(10 * fs8) / fs8
x8 = np.sin(2 * np.pi * 5 * t8) + 0.5 * np.sin(2 * np.pi * 60 * t8)
x8_100 = signal.resample_poly(x8, up=1, down=5)       # low-pass filter first, then keep every 5th sample

f8, P8 = signal.welch(x8_100, fs=100, nperseg=200)
plt.figure(figsize=(6, 2.5)); plt.semilogy(f8, P8)
plt.xlabel("frequency (Hz)"); plt.ylabel("PSD (a.u.²/Hz)"); plt.show()

In [ ]:
current = 8
results[8] = {}
try:
    fc, Pc = signal.welch(np.asarray(x8_100), fs=100, nperseg=200)
    band = lambda lo, hi: Pc[(fc >= lo) & (fc <= hi)].sum() * (fc[1] - fc[0])          # power in a band
    check(np.size(x8_100) == 1000, "1000 samples: 10 s at 100 Hz")
    check(band(38, 42) < 1e-3, "no 40 Hz alias")
    check(0.4 < band(4, 6) < 0.6, "the 5 Hz rhythm keeps its power (0.5 for amplitude 1)")
except NameError:
    results[8] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Takeaway:** In this example the new 40 Hz peak is an alias of 60 Hz. In general, a peak that appears after resampling calls for checking aliasing first, then the sampling-rate labels and other processing. Modules 4 §3c and 8 §3, and Chapter 2.

## Challenge 9 — A classifier that is too good ★★★

**Goal:** Estimate how well a classifier predicts the label for a new subject. The subject-wise splits `splits9` are given and correct. The features are random noise and the labels are random, so the **expected** accuracy is 50 %; one finite dataset can land a little above or below by chance. Keep the classifier in a variable called `model9`.

**Symptom:** It reports about 80 % accuracy on pure noise.

**How to troubleshoot it**

1. **Reproduce.** About 0.8 accuracy. Ask: *is this plausible?* The labels are random, so the expected accuracy on new subjects is 50 %. A result far above that is not automatically a bug — a small dataset can be lucky — but it is a strong reason to look for leakage before believing it.
2. **Find what saw the test data.** List every step that *learns* from data. Here `SelectKBest` picks features using `y9` — the labels of *all* epochs, before the cross-validation splits them.
3. **Cause.** The test folds' labels helped choose the features, so the 'best' 10 noise features happen to fit those very labels. The splits themselves were fine.
4. **Fix:** put the selector into a `Pipeline` with the classifier and pass the **raw** features to `cross_val_score`, so selection is re-fitted inside each training fold. The accuracy drops to about 0.5. The check looks at the structure (is selection inside the model?) as well as the number, because a number in the right range can also come from the wrong code.

**Fixed code**

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.pipeline import Pipeline, make_pipeline

rng9 = np.random.default_rng(2013)
X9 = rng9.standard_normal((120, 500))               # 120 epochs x 500 noise features
y9 = rng9.integers(0, 2, 120)                        # random labels
groups9 = np.repeat(np.arange(12), 10)              # 12 subjects, 10 epochs each
splits9 = list(GroupKFold(4).split(X9, y9, groups9))   # given, and correct: by subject

model9 = make_pipeline(SelectKBest(f_classif, k=10), LogisticRegression(max_iter=1000))   # selection inside
acc9 = cross_val_score(model9, X9, y9, cv=splits9).mean()
print(f"accuracy: {acc9:.2f}")

In [ ]:
current = 9
results[9] = {}
try:
    steps9 = [type(s).__name__ for _, s in model9.steps] if isinstance(model9, Pipeline) else []
    check("SelectKBest" in steps9, "feature selection is a step of model9, so it is re-fitted inside every fold")
    check("SelectKBest" in steps9 and np.isclose(cross_val_score(model9, X9, y9, cv=splits9).mean(), acc9),
          "acc9 comes from model9 on the raw features with the given splits")
    check(0.3 < acc9 < 0.7, "accuracy close to the expected 50 % (a diagnostic, not the proof)")
except NameError:
    results[9] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Takeaway:** Anything that learns from data (scalers, selection, resampling, tuning) belongs inside the cross-validation. Modules 5 §7 and 8 §4, Chapter 12.

## Challenge 10 — The wrong split unit ★★★

**Goal:** Each of 30 patients either has a condition or not (one label per patient), and contributes 10 epochs. Estimate how well a model recognises the condition in a **new patient**. Keep the splits you use in a variable called `splits10`.

**Symptom:** It reports about 96 % accuracy — better than seems realistic for this weak signal.

**How to troubleshoot it**

1. **Reproduce.** About 0.96. Plausible? The condition's effect is weaker than the differences between patients, so recognising it in a *new* patient should be much harder.
2. **Inspect the splits.** For the first fold, `set(patients10[tr]) & set(patients10[te])` is not empty: almost every patient has epochs on both sides.
3. **Cause.** `KFold` splits *epochs* at random. Every test patient was also seen in training, and because all of a patient's epochs share one label, the forest can recognise the *patient* (their fingerprint) and look up the label. That answers "new epochs of known patients", not the question asked.
4. **Fix:** split by patient with `GroupKFold` and the patient IDs. The accuracy drops to about 0.68 — the honest estimate for a new patient, and the one to report.

**Fixed code**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, KFold, GroupKFold

rng10 = np.random.default_rng(2013)
patients10 = np.repeat(np.arange(30), 10)                         # 30 patients x 10 epochs
y10 = np.array([0, 1] * 15)[patients10]                           # one label per patient
fingerprint = rng10.normal(0, 1.0, (30, 6))                       # every patient's signals look a bit different
X10 = fingerprint[patients10] + rng10.normal(0, 0.3, (300, 6))
X10[:, 0] += 1.5 * y10                                            # the real, weaker effect of the condition

model10 = RandomForestClassifier(n_estimators=200, random_state=0)
splits10 = list(GroupKFold(5).split(X10, y10, patients10))        # split by PATIENT
acc10 = cross_val_score(model10, X10, y10, cv=splits10).mean()
print(f"accuracy: {acc10:.2f}")

In [ ]:
current = 10
results[10] = {}
try:
    disjoint = all(set(patients10[tr]).isdisjoint(patients10[te]) for tr, te in splits10)
    covered = sorted(np.concatenate([te for _, te in splits10]).tolist()) == list(range(len(y10)))
    check(disjoint, "no patient is in both the training and the test part of any fold")
    check(covered, "every epoch is tested exactly once")
    check(disjoint and np.isclose(cross_val_score(model10, X10, y10, cv=splits10).mean(), acc10),
          "acc10 comes from model10 with these splits")
except NameError:
    results[10] = None
    print("NOT YET  the challenge cell above didn't finish: fix it and run it first")

**Takeaway:** The split unit must match the question: new patient → split by patient. Modules 5 §4 and 8 §4, Chapter 12, and the split unit in every capstone track card.

**Think further (optional).** There are 300 epochs but only 30 patients. Which number tells you how much
independent evidence you have about *new patients*? Try averaging the predictions per patient and scoring one
prediction per patient: how does that compare with the epoch-level accuracy?

There is no single "right" score here: 0.68 is what this split, seed and model give. Leave-one-patient-out or
another grouping is just as valid, as long as no patient is on both sides of a split.

## Summary

In [ ]:
EXPECTED = {1: 1, 2: 3, 3: 2, 4: 2, 5: 2, 6: 2, 7: 2, 8: 3, 9: 3, 10: 3}     # number of checks per challenge
total = sum(EXPECTED.values())
passed = 0
for i, n in EXPECTED.items():
    r = results.get(i, "not run")
    if r == "not run":
        status = "not checked yet"
    elif r is None:
        status = "code didn't finish"
    else:
        passed += sum(r.values())
        status = f"{sum(r.values())} of {n} pass"
    print(f"Challenge {i:2d}: {status}")
print(f"\n{passed} of {total} checks pass.")
print("This reflects the last time each check cell ran. After editing an answer, re-run its check cell.")

Back to the [overview](README.md).